# Day 29: Player Similarity Engine — Packaging, CLI & Production Documentation

**Phase 2: EDA & Feature Engineering | Project 1: FootballScout**

## Production Engineering & Reproducibility Audit

A machine learning data product in professional sports analytics has zero commercial value if coaching staffs and recruitment directors cannot reproduce its outputs, integrate it into automated reporting pipelines, or explore it through an intuitive interface.

### Objectives for Day 29:
1. **Production Module Packaging:** Upgrade `src/models/player_similarity.py` with clean PEP 257 docstrings, defensive input validation, and a command-line interface (`argparse`) supporting position filters, thresholding, and automated figure export (`reports/figures/`).
2. **Technical Documentation (README):** Author `docs/player_similarity_readme.md` detailing the problem statement, mathematical formulations, and empirical findings.
3. **Web Portfolio Product Integration:** Deploy `docs/footballscout.html`, harmonized with the portfolio design system (`docs/assets/css/style.css`), featuring a live zero-backend client-side scouting console.
4. **End-to-End Verification Workbook:** Validate both Python API and CLI executions directly in this notebook to guarantee zero-defect reproducibility.


## 1. Environment Setup & Core Engine Verification

We verify that the repository paths resolve accurately, load the tournament feature store, and initialize `PlayerSimilarityEngine` from `src.models.player_similarity`.


In [ ]:
import sys
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Resolve repository paths defensively
current_path = Path.cwd().resolve()
if (current_path / "src").exists():
    repo_root = current_path
elif (current_path / ".." / "..").resolve().joinpath("src").exists():
    repo_root = (current_path / ".." / "..").resolve()
else:
    repo_root = next((p for p in current_path.parents if (p / "src").exists()), current_path)

if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from src.models.player_similarity import PlayerSimilarityEngine

# 1. Explicitly define path to processed dataset
data_path = repo_root / "data" / "processed" / "football" / "player_per90_metrics.parquet"
assert data_path.exists(), f"Error: Dataset not found at {data_path}"

# 2. Load candidate players dataset
df_players = pd.read_parquet(data_path)

# 3. Initialize and fit the similarity engine
engine = PlayerSimilarityEngine(scaler_type="standard")
engine.fit(df_players)

print(f"[OK] Dataset successfully loaded from: {data_path.relative_to(repo_root)}")
print(f"[OK] PlayerSimilarityEngine initialized and fitted.")
print(f"Total qualified players: {len(df_players):,}")
print(f"Feature vector shape: {engine.X_scaled.shape[0]} players x {engine.X_scaled.shape[1]} metrics")


## 2. Programmatic Python API Execution

We test programmatic similarity queries via the Python API for:
1. **Luka Modrić** (Strict positional match: identifying creative midfield conductors).
2. **Kylian Mbappé** (Position-agnostic search: identifying explosive transition threats).


In [ ]:
# Case 1: Luka Modric (Midfielders Only)
print("=== PROGRAMMATIC SEARCH: Luka Modric (Strict Positional Cohort) ===")
results_modric = engine.find_similar("Luka Modrić", top_n=5, same_position=True, min_minutes=200.0)
display(results_modric[['player_name', 'team_name', 'primary_position', 'similarity_score', 'total_minutes_played', 'passes_completed_p90', 'progressive_passes_p90', 'xg_p90']])


In [ ]:
# Case 2: Kylian Mbappe (Any Position)
print("=== PROGRAMMATIC SEARCH: Kylian Mbappe (Position-Agnostic Cohort) ===")
results_mbappe = engine.find_similar("Kylian Mbappé", top_n=5, same_position=False, min_minutes=200.0)
display(results_mbappe[['player_name', 'team_name', 'primary_position', 'similarity_score', 'total_minutes_played', 'xg_p90', 'shots_p90', 'dribbles_completed_p90']])


## 3. Headless CLI Interface Verification

In production scouting environments, analysts execute automated batch jobs via the terminal. We test the CLI interface using `subprocess` and verify that command flags work seamlessly.


In [ ]:
python_exe = sys.executable
script_path = repo_root / "src" / "models" / "player_similarity.py"

# Test 1: Luka Modric search with automated plot saving
cmd_modric = [
    python_exe, str(script_path),
    "--player", "Luka Modrić",
    "--top-n", "5",
    "--same-position",
    "--min-minutes", "200",
    "--save-plots"
]

res_modric = subprocess.run(cmd_modric, cwd=str(repo_root), capture_output=True, text=True)
print("=== CLI STDOUT (Luka Modric) ===")
print(res_modric.stdout)
assert res_modric.returncode == 0, f"Modric CLI search failed: {res_modric.stderr}"


In [ ]:
# Test 2: Kylian Mbappe search (any position)
cmd_mbappe = [
    python_exe, str(script_path),
    "--player", "Kylian Mbappé",
    "--any-position",
    "--top-n", "5",
    "--save-plots"
]

res_mbappe = subprocess.run(cmd_mbappe, cwd=str(repo_root), capture_output=True, text=True)
print("=== CLI STDOUT (Kylian Mbappe) ===")
print(res_mbappe.stdout)
assert res_mbappe.returncode == 0, f"Mbappe CLI search failed: {res_mbappe.stderr}"


In [ ]:
# Test 3: Robust substring search (e.g. 'Messi')
cmd_fuzzy = [
    python_exe, str(script_path),
    "--player", "Messi",
    "--top-n", "3"
]

res_fuzzy = subprocess.run(cmd_fuzzy, cwd=str(repo_root), capture_output=True, text=True)
print("=== CLI STDOUT (Substring: 'Messi') ===")
print(res_fuzzy.stdout)
assert res_fuzzy.returncode == 0, f"Search failed: {res_fuzzy.stderr}"


## 4. Visual Artifact Verification (300 DPI Exports)

We verify that high-resolution visual reports have been persisted to `reports/figures/` and `docs/images/` for whitepaper and portfolio integration.


In [ ]:
figures_dir = repo_root / "reports" / "figures"
docs_img_dir = repo_root / "docs" / "images"

expected_figures = [
    "player_similarity_modric.png",
    "head_to_head_modric.png",
    "player_similarity_mbappe.png",
    "tactical_universe_2d.png"
]

print("=== Artifact Audit ===")
for fig_name in expected_figures:
    p_reports = figures_dir / fig_name
    p_docs = docs_img_dir / fig_name
    
    status_reports = f"EXISTS ({p_reports.stat().st_size / 1024:.1f} KB)" if p_reports.exists() else "MISSING"
    status_docs = f"EXISTS ({p_docs.stat().st_size / 1024:.1f} KB)" if p_docs.exists() else "MISSING"
    
    print(f"{fig_name:<30} | reports/figures: {status_reports:<20} | docs/images: {status_docs}")


In [ ]:
# Display inline preview of Modric similarity ranking
img_path = docs_img_dir / "player_similarity_modric.png"
if img_path.exists():
    display(Image(filename=str(img_path), width=750))


## 5. Web Portfolio & Client Data Verification

We verify the integration of our interactive web application in `docs/footballscout.html` and its client payload in `docs/assets/data/players_scouting.js`.


In [ ]:
html_path = repo_root / "docs" / "footballscout.html"
js_data_path = repo_root / "docs" / "assets" / "data" / "players_scouting.js"
index_path = repo_root / "docs" / "index.html"
readme_path = repo_root / "docs" / "player_similarity_readme.md"

print(f"FootballScout Page:      {'EXISTS' if html_path.exists() else 'MISSING'} ({html_path.stat().st_size / 1024:.1f} KB)")
print(f"Client Payload (JS):     {'EXISTS' if js_data_path.exists() else 'MISSING'} ({js_data_path.stat().st_size / 1024:.1f} KB)")
print(f"Central Portfolio Index: {'EXISTS' if index_path.exists() else 'MISSING'} ({index_path.stat().st_size / 1024:.1f} KB)")
print(f"Technical Whitepaper:    {'EXISTS' if readme_path.exists() else 'MISSING'} ({readme_path.stat().st_size / 1024:.1f} KB)")

# Verify link from index.html to footballscout.html
with open(index_path, "r", encoding="utf-8") as f:
    index_content = f.read()

assert "footballscout.html" in index_content, "footballscout.html link missing from index.html!"
print("\n[SUCCESS] Central portfolio index.html correctly routes to FootballScout.")


## 6. Day 29 Deliverables & Acceptance Criteria

| Component | Target Deliverable | Verification Status |
| :--- | :--- | :---: |
| **CLI Module** | `src/models/player_similarity.py` with `argparse` & error handling | **PASSED** |
| **Visual Exports** | 300 DPI tactical charts in `reports/figures/` & `docs/images/` | **PASSED** |
| **Technical Whitepaper** | `docs/player_similarity_readme.md` with math formulas & empirical tables | **PASSED** |
| **Interactive Portfolio** | `docs/footballscout.html` with client-side cosine engine | **PASSED** |
| **Site Navigation** | Project card in `docs/index.html` routed to `footballscout.html` | **PASSED** |
| **Workbook Audit** | Full execution reproducibility via `day029_project_1_readme.ipynb` | **PASSED** |

---
**Next Step (Day 30):** Consolidate the Phase 2 Feature Store and begin the NFL Draft Combine EDA & Feature Engineering project!
